<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/08_2_%EA%B7%9C%EC%B9%99_%EA%B8%B0%EB%B0%98_%EB%B6%84%EA%B8%B0_%EC%97%90%EC%9D%B4%EC%A0%84%ED%8A%B8_%EA%B5%AC%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 08-2: 규칙 기반(Rule-based) 분기 에이전트 구현  

### 실습목표

- 사용자의 입력 키워드를 분석하여 규칙 기반(Rule-based) 응답과 LLM 기반 응답으로 경로를 나누는 라우팅(Routing) 로직을 구현할 수 있다.  

- 정해진 비즈니스 규칙(FAQ)을 우선적으로 처리함으로써 답변의 정확성과 통제력을 확보하는 방법을 습득한다.  

- 08-1강에서 배운 Dialog State와 결합하여, 특정 조건에서만 동작하는 안정적인 에이전트 구조를 설계한다.  

### 1. 환경 준비
필요한 라이브러리를 설치합니다.

In [3]:
# 기존 설치를 무시하고 최신 버전으로 강제 재설치합니다.
!pip install -q -U --force-reinstall langchain langchain-community langchain-huggingface langchain-core langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.1.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.1 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.1 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.1 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.1.0 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.1 wh

### 2. Gemini LLM 로드

Google Gemini 모델을 사용하여 더 나은 답변을 시도할 수 있습니다. Gemini 모델을 사용하려면 `google-generativeai` 라이브러리를 설치하고 API 키를 설정해야 합니다.

In [4]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API 설정 완료")

Gemini API 설정 완료


3. 규칙 기반 분기 처리의 구조  

- 단순히 모든 질문을 LLM에 던지는 것이 아니라, Keyword Matching을 통해 답변의 경로를 결정합니다.  

    - Step 1: 사용자 입력에서 특정 키워드(예: '환불', '배송', '시간') 탐지  

    - Step 2: 키워드가 존재할 경우, 미리 정의된 고정 답변(Fixed Response) 출력  

    - Step 3: 키워드가 없을 경우에만 LLM 기반 대화 체인으로 연결  

In [7]:
import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_community.chat_message_histories import ChatMessageHistory

# --- 1. 상태 및 규칙 정의 ---
history_db = ChatMessageHistory()
dialog_state = {"slots": {"name": None, "location": None, "topic": None}}

# 비즈니스 규칙(FAQ) 데이터베이스 정의
FAQ_RULES = {
    "시간": "저희 상담소 운영 시간은 평일 오전 9시부터 오후 6시까지입니다.",
    "위치": "저희 사무실은 서울시 강남구 테헤란로에 위치해 있습니다.",
    "연락처": "문의사항은 02-123-4567로 전화 주시면 친절히 안내해 드립니다."
}

# --- 2. LLM 및 프롬프트 설정 ---
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY)

template = """너는 사용자의 정보 수집을 돕는 친절한 상담사야.
파악된 정보: 이름({name}), 지역({location}), 주제({topic})
위 정보를 바탕으로 부족한 내용을 물어보거나 대화를 이어가줘.

[대화 기록]
{history}
사용자: {question}
답변:"""

prompt = PromptTemplate.from_template(template)

# --- 3. LCEL 체인 구성 ---
lcel_chain = (
    {
        "question": RunnablePassthrough(),
        "history": RunnableLambda(lambda _: "\n".join([f"{m.type}: {m.content}" for m in history_db.messages])),
        "name": lambda _: dialog_state["slots"]["name"] or "미파악",
        "location": lambda _: dialog_state["slots"]["location"] or "미파악",
        "topic": lambda _: dialog_state["slots"]["topic"] or "미파악"
    }
    | prompt | llm | StrOutputParser()
)

# --- 4. [핵심] 규칙 기반 분기 로직 (Rule-based Routing) ---
def chat(user_input):
    # 1. 규칙 기반 체크 (Rule-based Matching)
    for key, fixed_answer in FAQ_RULES.items():
        if key in user_input:
            print(f"[*] 시스템: 규칙 기반(FAQ) 응답이 트리거되었습니다.") #
            history_db.add_user_message(user_input)
            history_db.add_ai_message(fixed_answer)
            return fixed_answer

    # 2. 규칙에 해당하지 않을 경우 LLM 실행
    response = lcel_chain.invoke(user_input)

    # [08-1 개념] 슬롯 업데이트 로직 병행
    if "제미니" in user_input: dialog_state["slots"]["name"] = "제미니"
    if "수원" in user_input: dialog_state["slots"]["location"] = "수원"

    history_db.add_user_message(user_input)
    history_db.add_ai_message(response)
    return response

print("규칙 기반 분기 에이전트 로드 완료!")

규칙 기반 분기 에이전트 로드 완료!


3. 실습 테스트 및 검증  

- 에이전트가 키워드에 따라 어떻게 다르게 반응하는지 확인합니다.  

In [8]:
# --- [08-2 테스트 시나리오] ---

# 테스트 1: 규칙 기반(FAQ) 분기 확인
print("Q: 너희 사무실 위치가 어디야?")
print(f"A: {chat('너희 사무실 위치가 어디야?')}\n")

# 테스트 2: LLM 기반 자유 대화 및 슬롯 필링 확인
print("Q: 안녕, 내 이름은 제미니야.")
print(f"A: {chat('안녕, 내 이름은 제미니야.')}\n")

Q: 너희 사무실 위치가 어디야?
[*] 시스템: 규칙 기반(FAQ) 응답이 트리거되었습니다.
A: 저희 사무실은 서울시 강남구 테헤란로에 위치해 있습니다.

Q: 안녕, 내 이름은 제미니야.
A: 안녕하세요, 제미니님! 만나 뵙게 되어 정말 반갑습니다. 멋진 이름이시네요. 😊

저희 사무실 위치는 확인되셨는데, 혹시 어떤 주제에 대한 정보 수집을 도와드리면 될까요?

제미니님께서 찾으시는 정보의 종류나, 계신 지역(예: 서울, 부산 등)을 알려주시면 더 자세하고 정확한 내용을 안내해 드릴 수 있습니다. 편하게 말씀해주세요!



#### 보강 포인트

- Fallback 방지: LLM이 답변하기 까다로운 정적인 정보(주소, 전화번호 등)를 Fixed Response로 처리하여 정확도를 높였습니다.  

- 통제력 확보: 비즈니스 운영 시간과 같은 민감한 정보는 LLM의 할루시네이션(환각) 없이 규칙에 따라 100% 정확하게 출력됩니다.  

- 유기적 결합: 규칙에 해당하지 않는 '일상적인 인사'나 '복잡한 맥락'은 다시 LLM의 지능으로 처리하여 유연성을 잃지 않았습니다.  